In [78]:
# 列出每个通过审核case的详细信息
if approved_cases:
    print(f"\n📋 通过审核Case详细信息:")
    print("-" * 85)
    print(f"{'编号':<25}{'申请人':<15}{'客户':<28}{'用时(天)':<10}")
    print("-" * 85)
    for case in approved_cases:
        id_short = case['编号'][:23] + '..' if len(case['编号']) > 25 else case['编号']
        applicant_short = case['申请人'][:13] + '..' if len(case['申请人']) > 15 else case['申请人']
        customer = case['客户']
        # 检查是否包含中文
        has_chinese = any('\u4e00' <= c <= '\u9fff' for c in customer)
        if has_chinese:
            # 中文字符截断到6个字符
            if len(customer) > 8:
                customer_short = customer[:6] + '..'
            else:
                customer_short = customer
        else:
            # 英文字符截断到26个字符
            if len(customer) > 28:
                customer_short = customer[:26] + '..'
            else:
                customer_short = customer
        duration_short = str(case['用时(天)'])
        print(f"{id_short:<25}{applicant_short:<15}{customer_short:<28}{duration_short:<10}")

print("=" * 85)


📋 通过审核Case详细信息:
-------------------------------------------------------------------------------------
编号                       申请人            客户                          用时(天)     
-------------------------------------------------------------------------------------
FV-20260108-49709        yugu           Cadence Design Systems, Inc.6         
FV-20260126-50128        qingyan        XIAO JU SCIENCE AND TECHNO..2         
FV-20260129-50201        lilyzheng1     Innov-Tech Prime Consultan..13        
FV-20260307-50842        annazeng       AppLovin                    5         
FV-20260310-50909        yugu           河北汇鲸珑腾..                    2         
FV-20260319-51126        yugu           SECURITY NETWORK INTELLIGE..20        
FV-20260409-51542        christineluo   SANHUA AUTOMOTIVE USA，LLC   1         
FV-20260421-51754        qingyan        Joyalways Investment Corpo..14        
FV-20260423-51823        robertsong     ATTO DIGITAL INFORMATION T..11        
FV-20260429-51939    

In [74]:
# ============================================
# 功能说明：押金相关Case统计与跟踪
# ============================================
# 1. 识别明确要求缴纳押金的case（排除豁免和建议性语言）
# 2. 检查是否已上传押金凭证（财务部确认到款）
# 3. 提取销售信息（销售姓名、销售部门、客户名称）
# 4. 显示财务要求缴纳押金的日期
# 5. 显示所有押金case的详细信息
# 6. 标注需要跟进的case（未上传押金凭证）
# ============================================

# 押金相关统计
deposit_cases = []
deposit_with_receipt = []
deposit_without_receipt = []

for app_id in all_ids:
    app_df = df[df['参与方验证编码'] == app_id]
    finance_1st_all = app_df[app_df['信审审批节点名称'] == '03 财务一级审核']
    
    is_require_deposit = False
    head_opinion = '无'
    deposit_date = None
    
    if len(finance_1st_all) > 0:
        for idx, row in finance_1st_all.iterrows():
            opinion_1st = str(row['审批意见']).lower() if pd.notna(row['审批意见']) else ''
            is_exempt = '豁免押金' in opinion_1st or '免押金' in opinion_1st
            
            if (
                ('客户需缴纳押金' in opinion_1st or 
                 '需缴纳押金' in opinion_1st or 
                 '要求缴纳押金' in opinion_1st or
                 '在收到' in opinion_1st and '押金' in opinion_1st) and not is_exempt
            ):
                is_require_deposit = True
                head_opinion = row['审批意见']
                # 获取财务要求缴纳押金的日期
                deposit_date = row['审批开始时间'] if pd.notna(row['审批开始时间']) else None
                break
        
        if not is_require_deposit:
            head_opinion = finance_1st_all.iloc[0]['审批意见']
        
        if is_require_deposit:
            has_receipt = len(app_df[app_df['信审审批节点名称'] == '18 财务部（合同组）确认到款']) > 0
            sales_name = app_df['信审申请人'].iloc[0] if '信审申请人' in app_df.columns else '未知'
            sales_dept = app_df['信审申请部门'].iloc[0] if '信审申请部门' in app_df.columns else '未知'
            customer = app_df['信审客户-iBOSS名称'].iloc[0] if '信审客户-iBOSS名称' in app_df.columns else '未知'
            
            # 格式化日期
            deposit_date_str = deposit_date.strftime('%Y-%m-%d') if deposit_date is not None else '未知'
            
            deposit_info = {
                '编号': app_id,
                '销售': sales_name,
                '销售部门': sales_dept,
                '客户': customer,
                '押金要求日期': deposit_date_str,
                '总部意见': str(head_opinion)[:100] + '...' if len(str(head_opinion)) > 100 else str(head_opinion),
                '已上传凭证': '是' if has_receipt else '否'
            }
            deposit_cases.append(deposit_info)
            
            if has_receipt:
                deposit_with_receipt.append(deposit_info)
            else:
                deposit_without_receipt.append(deposit_info)

# 显示押金统计
print("=" * 80)
print("押金相关Case统计")
print("=" * 80)
print(f"明确要求缴纳押金的Case总数: {len(deposit_cases)}")
print(f"已上传押金凭证: {len(deposit_with_receipt)}")
print(f"未上传押金凭证: {len(deposit_without_receipt)}")

if deposit_cases:
    print(f"\n📋 所有押金Case详细信息:")
    print("-" * 80)
    for i, case in enumerate(deposit_cases, 1):
        print(f"\n【Case {i}】")
        print(f"  编号: {case['编号']}")
        print(f"  销售: {case['销售']} ({case['销售部门']})")
        print(f"  客户: {case['客户']}")
        print(f"  押金要求日期: {case['押金要求日期']}")
        print(f"  上传状态: {case['已上传凭证']}")
        if case['已上传凭证'] == '否':
            print(f"  ⚠️  需要跟进！")
        print(f"  总部意见: {case['总部意见']}")
else:
    print("\n没有需要缴纳押金的Case")

print("=" * 80)

押金相关Case统计
明确要求缴纳押金的Case总数: 2
已上传押金凭证: 1
未上传押金凭证: 1

📋 所有押金Case详细信息:
--------------------------------------------------------------------------------

【Case 1】
  编号: FV-20260423-51823
  销售: robertsong (美国公司-企业业务)
  客户: ATTO DIGITAL INFORMATION TECHNOLOGY (HONG KONG) LIMITED
  押金要求日期: 2026-04-27
  上传状态: 否
  ⚠️  需要跟进！
  总部意见: 根据第三方信用报告评估结果，建议客户为中风险，在收到不少于50%的一次性费用及相等于2个月月费的押金(可接受形式有：现金、Traffic Prepayment)之后，授予客户相等于2个月月费的信用额开...

【Case 2】
  编号: FV-20260606-52662
  销售: christineluo (美国公司-企业业务)
  客户: 北京数信网联信息科技有限公司
  押金要求日期: 2026-06-11
  上传状态: 是
  总部意见: 客户需缴纳押金才能开展业务，外汇管制风险可控，经核对，拟同意
